# Dynamic Source Routing in Ad Hoc Wireless Networks

**David B. Johnson and David A. Maltz, chapter version prepared for _Mobile Computing_ (1996)**

This notebook is a self-contained reproduction of the DSR paper.
It recreates the protocol diagrams in Figures 1–4, generates a representative disconnected-cluster layout for Figure 6,
and reproduces the performance trends in Figures 5 and 7 with an explicitly documented aggregate simulator.

> **Citation**
> Johnson, D. B., & Maltz, D. A. (1996).
> *Dynamic Source Routing in Ad Hoc Wireless Networks*.
> In *Mobile Computing*.


## Algorithm Explanation

Dynamic Source Routing (DSR) is an **on-demand source-routing protocol** for mobile ad hoc networks.
A sender places the entire hop sequence in the packet header, so every forwarding node only needs to read the next hop.
The protocol is organized around two mechanisms:

1. **Route discovery.**
   If source node $s$ needs a route to destination $d$ and has no cached route, it broadcasts a route request
   carrying the tuple $(s, d, \text{request\_id}, \text{route\_record})$.
   Each receiver:
   - discards duplicates of the same $(s, \text{request\_id})$,
   - discards requests whose route record already contains itself,
   - replies if it is the target or if its route cache already contains a loop-free route to the target,
   - otherwise appends itself to the route record and rebroadcasts.

   If a request reaches a responder with accumulated record
   $R = (v_0 = s, v_1, \ldots, v_k)$
   and the responder knows a cached suffix
   $Q = (v_k, v_{k+1}, \ldots, v_m = d)$,
   then the returned source route is the concatenation
   $$
   R \oplus Q = (v_0, v_1, \ldots, v_k, v_{k+1}, \ldots, v_m).
   $$

2. **Route maintenance.**
   While a route
   $$
   P = (u_0 = s, u_1, \ldots, u_h = d)
   $$
   is in use, each hop relies on link-layer acknowledgements to detect a failure of $(u_i, u_{i+1})$.
   When a hop fails, the detecting node returns a route error toward the source, and every cached route containing the failed hop is truncated or removed.

### Optimizations in the paper

- **Full use of the route cache.**
  Nodes learn routes from route requests, route replies, and forwarded data packets.
  The cache behaves like a learned graph of usable hops.
- **Delayed cache replies.**
  To prefer shorter cached routes and suppress collisions, the paper delays cache replies by
  $$
  d = H (h - 1 + r),
  $$
  where $h$ is the route length in hops, $r \sim \mathrm{Uniform}(0,1)$, and $H$ is the per-hop holdoff constant.
- **One-hop nonpropagating route request.**
  A sender first tries a hop-limited request before flooding the whole network.
- **Piggybacking.**
  Small control payloads can be attached to route discoveries.
- **Route shortening.**
  Promiscuous overhearing can reveal that an intermediate node became unnecessary.
  The paper describes this optimization but explicitly says it was **not** included in the simulation.
- **Improved error handling.**
  Exponential backoff suppresses repeated discoveries to unreachable targets, and overheard route errors invalidate failed hops in nearby caches.

### What the evaluation measures

Figure 5 reports
$$
\frac{\text{total packet transmissions actually performed}}{\text{optimal number of data-packet transmissions}},
$$
where the numerator includes data, route requests, route replies, and route errors.

Figure 7 reports
$$
\frac{\text{average source-route length used}}{\text{optimal route length}},
$$
showing how close DSR stays to hop-optimal routing while nodes move.


## Explicit Assumptions

The chapter gives the protocol clearly, but several details needed for an executable notebook are omitted or not machine-readable from the scan.

| # | Underspecified Detail | Assumption Used Here |
|---|---|---|
| 1 | Notebook filename year | `1996`, matching the chapter version announced in the PDF. |
| 2 | Exact pause-time sample points used in Figures 5 and 7 | Approximated as $[0, 30, 60, 120, 300, 600, 1000, 2000, 4000]$ seconds because the text gives only the range $0$ to $4000$. |
| 3 | Cache-reply holdoff equation in the scanned PDF | Use $d = H (h - 1 + r)$ from the 1998 DSR Internet-Draft, which matches the surrounding prose and later DSR constants. |
| 4 | Route-cache expiration time | Use `300 s` from later DSR drafts; the chapter says cache entries expire but gives no value. |
| 5 | Route reply and route error return path in the simulator | Use the reverse path, because the paper's simulator assumes symmetric links through link-layer acknowledgements and does not model one-way links. |
| 6 | Packet sizes and bandwidth effect on topology change during one send | Packet sizes are sampled for completeness, but transmission-time-induced mid-packet motion is ignored because the paper evaluates transmission counts, not latency. |
| 7 | Link-layer retries in the aggregate model | Replace per-packet retries by the expected attempt count $1 + p + p^2 = 1.0525$ for in-range unicast links with loss probability $p = 0.05$. |
| 8 | Pure-Python runtime budget for the multi-run sweep | Use a documented aggregate simulator with `5 s` time quanta, `1000 s` simulated duration, and `5` runs per point instead of the paper's `4000 s` and `20` runs; pause times at or above `1000 s` therefore appear nearly static in this model. |
| 9 | Cache invalidation after a detected broken hop | Apply the failed-hop invalidation to every cache, representing the optimistic promiscuous route-error handling described in Section 4.4. |
| 10 | Figure 6 random placement | Search deterministic seeds and select the first 24-host placement that produces exactly two connected components. |
| 11 | Return traffic | Every successfully delivered forward packet generates one reverse-direction packet along the reverse source route, matching the paper's "two way conversation" model. |
| 12 | Unreachable destinations in normalized metrics | If no path exists in the instantaneous topology, control overhead is counted, but optimal hop denominators are not incremented because the paper does not define a finite optimal path in that case. |
| 13 | Paper-comparison numbers for Figures 5 and 7 | The black comparison curves are approximate digitizations read from the printed plots and are used only to quantify reproduction error. |


In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(linewidth=140, suppress=True)

PAUSE_TIMES = np.array([0.0, 30.0, 60.0, 120.0, 300.0, 600.0, 1000.0, 2000.0, 4000.0], dtype=float)
HOST_COUNTS = np.array([6, 12, 18, 24], dtype=int)
MARKERS = {6: "s", 12: "o", 18: "^", 24: "D"}
LINESTYLES = {6: "-", 12: ":", 18: "-.", 24: "--"}
COLORS = {6: "#2f4858", 12: "#33658a", 18: "#86bbd8", 24: "#f26419"}

PAPER_FIGURE5 = np.array(
    [
        [1.30, 1.20, 1.17, 1.14, 1.12, 1.08, 1.08, 1.13, 1.10],
        [1.58, 1.32, 1.23, 1.18, 1.12, 1.08, 1.06, 1.06, 1.01],
        [1.99, 1.55, 1.40, 1.27, 1.14, 1.09, 1.07, 1.01, 1.02],
        [2.57, 1.88, 1.54, 1.34, 1.13, 1.08, 1.06, 1.01, 1.00],
    ],
    dtype=float,
)

PAPER_FIGURE7 = np.array(
    [
        [1.090, 1.064, 1.064, 1.043, 1.027, 1.012, 1.008, 1.011, 1.001],
        [1.078, 1.075, 1.055, 1.037, 1.024, 1.023, 1.013, 1.008, 1.001],
        [1.041, 1.056, 1.052, 1.033, 1.016, 1.014, 1.016, 1.005, 1.004],
        [1.028, 1.034, 1.035, 1.021, 1.014, 1.013, 1.012, 1.006, 1.003],
    ],
    dtype=float,
)


@dataclass(frozen=True)
class EvalConfig:
    """Configuration for the aggregate DSR evaluation model."""

    area_size: float = 9.0
    radio_range: float = 3.0
    min_speed: float = 0.3
    max_speed: float = 0.7
    duration: float = 1000.0
    step: float = 5.0
    trials: int = 5
    max_conversations_per_host: int = 3
    mean_conversation_interval: float = 15.0
    mean_conversation_packets: float = 1000.0
    min_packet_rate: float = 2.0
    max_packet_rate: float = 5.0
    link_failure_prob: float = 0.05
    route_cache_timeout: float = 300.0
    nonpropagating_request_period: float = 5.0
    max_request_period: float = 10.0


SIM_CONFIG = EvalConfig()
EXPECTED_UNICAST_ATTEMPTS = 1.0 + SIM_CONFIG.link_failure_prob + SIM_CONFIG.link_failure_prob**2
print(SIM_CONFIG)


## Figure 1 Explanation

Figure 1 introduces the basic reason routing is needed.
Hosts $A$ and $C$ are outside one another's transmission range, but host $B$ lies in the overlap of both coverage circles.
The expected qualitative result is three overlapping radio disks with only the middle node able to relay traffic between the endpoints.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

centers = {"A": (0.0, 0.0), "B": (1.4, 0.0), "C": (2.8, 0.0)}
circle_colors = {"A": "#8d99ae", "B": "#edf2f4", "C": "#cfd2d6"}

for label, (x, y) in centers.items():
    ax.add_patch(Circle((x, y), radius=1.55, fill=False, linewidth=3, linestyle="--", edgecolor=circle_colors[label]))
    ax.add_patch(Rectangle((x - 0.16, y - 0.16), 0.32, 0.32, facecolor="white", edgecolor="black", linewidth=2))
    ax.text(x, y, label, ha="center", va="center", fontsize=16, family="serif", style="italic")

ax.set_aspect("equal")
ax.set_xlim(-1.9, 4.5)
ax.set_ylim(-1.8, 1.8)
ax.axis("off")
ax.set_title("Figure 1 Reproduction: Three Wireless Mobile Hosts", fontsize=14)
plt.show()


## Figure 2 Explanation

Figure 2 illustrates **full use of the route cache**.
Node $A$ already knows the route $A \rightarrow B \rightarrow C \rightarrow D$,
learns that $E$ is reachable through the same prefix, and can answer a request from $F$ using cached state.
The layout is schematic rather than metric: what matters is the chain $A$-$B$-$C$-$D$, the nearby node $E$, and requester $F$ below the chain.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

positions = {
    "A": (0.0, 0.0),
    "B": (1.4, 0.0),
    "C": (2.8, 0.0),
    "D": (4.2, 0.0),
    "E": (3.7, -0.9),
    "F": (0.9, -1.1),
}

for start, end in [("A", "B"), ("B", "C"), ("C", "D")]:
    x0, y0 = positions[start]
    x1, y1 = positions[end]
    ax.add_patch(
        FancyArrowPatch(
            (x0 + 0.2, y0),
            (x1 - 0.2, y1),
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.6,
            linestyle=(0, (4, 4)),
            color="black",
        )
    )

for label, (x, y) in positions.items():
    ax.add_patch(Rectangle((x - 0.22, y - 0.22), 0.44, 0.44, facecolor="white", edgecolor="black", linewidth=2))
    ax.text(x, y, label, ha="center", va="center", fontsize=16, family="serif", style="italic")

ax.text(-0.25, 0.58, "B-C-D", fontsize=18, family="serif", style="italic")
ax.set_aspect("equal")
ax.set_xlim(-0.8, 5.0)
ax.set_ylim(-1.7, 1.0)
ax.axis("off")
ax.set_title("Figure 2 Reproduction: Route-Cache Example", fontsize=14)
plt.show()


## Figure 3 Explanation

Figure 3 shows the **route-shortening idea** that the paper discusses but does not simulate.
A packet is being forwarded along $B \rightarrow C \rightarrow D$.
If $D$ overhears a transmission directly from $B$, it can infer that $C$ is no longer necessary and send an unsolicited shorter route back to the source.
The expected qualitative result is a straight route with a curved shortcut from $B$ to $D$.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))

positions = {"B": (0.0, 0.0), "C": (1.4, 0.0), "D": (2.8, 0.0)}

ax.add_patch(
    FancyArrowPatch(
        (-0.9, 0.0),
        (-0.15, 0.0),
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.4,
        linestyle=(0, (4, 4)),
        color="black",
    )
)

for start, end in [("B", "C"), ("C", "D")]:
    x0, y0 = positions[start]
    x1, y1 = positions[end]
    ax.add_patch(
        FancyArrowPatch((x0 + 0.22, y0), (x1 - 0.22, y1), arrowstyle="-|>", mutation_scale=16, linewidth=1.5, color="black")
    )

ax.add_patch(
    FancyArrowPatch(
        (0.0, -0.22),
        (2.8, -0.22),
        connectionstyle="arc3,rad=0.33",
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.5,
        color="#8d99ae",
    )
)

ax.add_patch(
    FancyArrowPatch(
        (2.95, 0.0),
        (3.65, 0.0),
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.4,
        linestyle=(0, (4, 4)),
        color="black",
    )
)

for label, (x, y) in positions.items():
    ax.add_patch(Rectangle((x - 0.22, y - 0.22), 0.44, 0.44, facecolor="white", edgecolor="black", linewidth=2))
    ax.text(x, y, label, ha="center", va="center", fontsize=16, family="serif", style="italic")

ax.set_aspect("equal")
ax.set_xlim(-1.1, 3.9)
ax.set_ylim(-0.9, 0.8)
ax.axis("off")
ax.set_title("Figure 3 Reproduction: Route Shortening by Promiscuous Overhearing", fontsize=14)
plt.show()


## Figure 4 Explanation

Figure 4 shows **route error propagation**.
Host $B$ detects a failed hop and sends a route error back to $A$.
Nodes $C$, $D$, and $E$ are arranged around the error path because they may overhear the route error and invalidate the broken hop in their own caches.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

positions = {
    "A": (0.0, 0.0),
    "B": (2.0, 0.0),
    "C": (0.55, 1.0),
    "D": (1.55, 1.15),
    "E": (1.0, -1.0),
}

ax.add_patch(
    FancyArrowPatch(
        (1.8, 0.0),
        (0.2, 0.0),
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8,
        color="black",
    )
)
ax.text(1.0, 0.18, '"route error"', ha="center", va="bottom", fontsize=13, family="serif", style="italic")

for label, (x, y) in positions.items():
    ax.add_patch(Rectangle((x - 0.22, y - 0.22), 0.44, 0.44, facecolor="white", edgecolor="black", linewidth=2))
    ax.text(x, y, label, ha="center", va="center", fontsize=16, family="serif", style="italic")

ax.set_aspect("equal")
ax.set_xlim(-0.8, 2.8)
ax.set_ylim(-1.5, 1.6)
ax.axis("off")
ax.set_title("Figure 4 Reproduction: Route Error Returned to the Sender", fontsize=14)
plt.show()


## Figure 5 Explanation

Figure 5 plots **normalized transmission overhead** as a function of pause time for four different network sizes.
The paper's x-axis is pause time in seconds; the y-axis is the ratio
$\text{actual transmissions} / \text{optimal data transmissions}$.

Expected qualitative behavior:

- larger pause times mean less movement, so the ratio should approach $1$,
- constant motion ($0$ pause) should produce the highest overhead,
- sparse networks such as 6 or 12 hosts can still show elevated overhead at large pause times because random placements create disconnected components.

The next code cell defines the aggregate simulator used for Figures 5–7.


In [ ]:
@dataclass
class MobileNode:
    """One random-waypoint mobile host tracked at aggregate time resolution."""

    pos: np.ndarray
    paused: bool
    remaining: float
    target: np.ndarray | None = None
    speed: float = 0.0

    def advance(self, dt: float, pause_time: float, rng: np.random.Generator, cfg: EvalConfig) -> None:
        """Advance the node by `dt` seconds under the paper's pause/move model."""

        while dt > 1e-9:
            spend = min(dt, self.remaining)

            if not self.paused and self.target is not None and self.remaining > 0.0:
                # Linear interpolation is enough because the paper uses constant-speed waypoint motion.
                self.pos = self.pos + (self.target - self.pos) * (spend / self.remaining)

            self.remaining -= spend
            dt -= spend

            if self.remaining > 1e-9:
                break

            if self.paused:
                self.paused = False
                self.target = rng.uniform(0.0, cfg.area_size, size=2)
                self.speed = float(rng.uniform(cfg.min_speed, cfg.max_speed))
                distance = float(np.linalg.norm(self.target - self.pos))

                if distance < 1e-12:
                    self.paused = True
                    self.remaining = pause_time
                else:
                    self.remaining = distance / self.speed
            else:
                self.pos = self.target.copy()
                self.target = None
                self.paused = True
                self.remaining = pause_time


@dataclass
class Conversation:
    """A source-initiated conversation with automatic reverse-direction replies."""

    initiator: int
    partner: int
    rate: float
    remaining_packets: int
    route: tuple[int, ...] | None = None


class CacheGraph:
    """Per-node learned hop graph used as a route cache.

    DSR caches full source routes, but representing them as learned undirected edges makes it easy
    to concatenate information from requests, replies, and forwarded data.
    """

    def __init__(self, node_count: int) -> None:
        self.expiry = np.zeros((node_count, node_count), dtype=float)

    def add_path(self, path: list[int] | tuple[int, ...], time: float, cfg: EvalConfig) -> None:
        """Insert every hop on a learned source route."""

        for a, b in zip(path, path[1:]):
            self.expiry[a, b] = self.expiry[b, a] = max(self.expiry[a, b], time + cfg.route_cache_timeout)

    def remove_edge(self, a: int, b: int) -> None:
        """Invalidate a failed hop everywhere it appears in the cache graph."""

        self.expiry[a, b] = self.expiry[b, a] = 0.0

    def path(self, node: int, target: int, time: float) -> list[int] | None:
        """Recover the current shortest cached path from `node` to `target`."""

        graph = [
            [j for j in range(self.expiry.shape[0]) if i != j and self.expiry[i, j] >= time]
            for i in range(self.expiry.shape[0])
        ]
        return shortest_path(graph, node, target)


def init_nodes(host_count: int, pause_time: float, rng: np.random.Generator) -> list[MobileNode]:
    """Place hosts uniformly in the room and start them in the pause phase."""

    return [
        MobileNode(pos=rng.uniform(0.0, SIM_CONFIG.area_size, size=2), paused=True, remaining=pause_time)
        for _ in range(host_count)
    ]


def build_connectivity_graph(nodes: list[MobileNode], cfg: EvalConfig) -> list[list[int]]:
    """Build the instantaneous undirected communication graph from Euclidean range."""

    host_count = len(nodes)
    graph = [[] for _ in range(host_count)]

    for i in range(host_count):
        for j in range(i + 1, host_count):
            if float(np.linalg.norm(nodes[i].pos - nodes[j].pos)) <= cfg.radio_range:
                graph[i].append(j)
                graph[j].append(i)

    return graph


def shortest_path(graph: list[list[int]], source: int, target: int) -> list[int] | None:
    """Return the minimum-hop path in an unweighted graph using BFS."""

    queue = deque([source])
    prev: dict[int, int | None] = {source: None}

    while queue:
        node = queue.popleft()

        if node == target:
            break

        for neighbor in graph[node]:
            if neighbor not in prev:
                prev[neighbor] = node
                queue.append(neighbor)

    if target not in prev:
        return None

    path = [target]

    while path[-1] != source:
        path.append(prev[path[-1]])

    path.reverse()
    return path


def route_valid(route: tuple[int, ...], graph: list[list[int]]) -> bool:
    """Check whether every hop on a cached source route is currently present."""

    return all(route[i + 1] in graph[route[i]] for i in range(len(route) - 1))


def perform_route_discovery(
    source: int,
    target: int,
    graph: list[list[int]],
    caches: list[CacheGraph],
    time: float,
    cfg: EvalConfig,
) -> tuple[tuple[int, ...] | None, float]:
    """Run the paper's one-hop request followed by a full flood if needed.

    The control-transmission accounting is aggregate rather than packet-exact:
    broadcasts count once per forwarding node, and successful unicast replies count
    the expected number of link-layer attempts.
    """

    control_tx = 1.0
    responders: list[list[int]] = []

    for neighbor in graph[source]:
        request_path = [source, neighbor]
        caches[neighbor].add_path(request_path, time, cfg)
        cached_suffix = caches[neighbor].path(neighbor, target, time)

        if neighbor == target:
            responders.append(request_path)
        elif cached_suffix is not None:
            full_route = request_path + cached_suffix[1:]
            if len(set(full_route)) == len(full_route):
                responders.append(full_route)

    if responders:
        best = min(responders, key=len)
        control_tx += (len(best) - 1) * EXPECTED_UNICAST_ATTEMPTS

        for node in best:
            caches[node].add_path(best, time, cfg)

        return tuple(best), control_tx

    control_tx += 1.0
    seen = {source}
    queue = deque([[source]])
    responders = []

    while queue:
        request_path = queue.popleft()
        last = request_path[-1]

        if last != source:
            control_tx += 1.0

        for neighbor in graph[last]:
            if neighbor in seen or neighbor in request_path:
                continue

            seen.add(neighbor)
            new_path = request_path + [neighbor]
            caches[neighbor].add_path(new_path, time, cfg)
            cached_suffix = caches[neighbor].path(neighbor, target, time)

            if neighbor == target:
                responders.append(new_path)
            elif cached_suffix is not None:
                full_route = new_path + cached_suffix[1:]
                if len(set(full_route)) == len(full_route):
                    responders.append(full_route)
            else:
                queue.append(new_path)

    if not responders:
        return None, control_tx

    best = min(responders, key=len)
    control_tx += (len(best) - 1) * EXPECTED_UNICAST_ATTEMPTS

    for node in best:
        caches[node].add_path(best, time, cfg)

    return tuple(best), control_tx


def run_trial(host_count: int, pause_time: float, seed: int, cfg: EvalConfig) -> tuple[float, float]:
    """Run one aggregate DSR trial and return the two normalized paper metrics."""

    rng = np.random.default_rng(seed)
    nodes = init_nodes(host_count, pause_time, rng)
    caches = [CacheGraph(host_count) for _ in range(host_count)]
    next_conversation_attempt = rng.exponential(cfg.mean_conversation_interval, size=host_count)
    active: list[Conversation] = []
    next_discovery_time = np.zeros((host_count, host_count), dtype=float)
    request_period = np.full((host_count, host_count), cfg.nonpropagating_request_period, dtype=float)

    total_tx = 0.0
    optimal_tx = 0.0
    used_hops = 0.0
    optimal_hops = 0.0
    time = 0.0

    while time < cfg.duration:
        graph = build_connectivity_graph(nodes, cfg)
        initiator_counts = [0] * host_count

        for conv in active:
            initiator_counts[conv.initiator] += 1

        for host in range(host_count):
            while (
                next_conversation_attempt[host] <= time + 1e-9
                and initiator_counts[host] < cfg.max_conversations_per_host
            ):
                partner = int(rng.integers(0, host_count - 1))
                if partner >= host:
                    partner += 1

                active.append(
                    Conversation(
                        initiator=host,
                        partner=partner,
                        rate=float(rng.uniform(cfg.min_packet_rate, cfg.max_packet_rate)),
                        remaining_packets=int(rng.geometric(1.0 / cfg.mean_conversation_packets)),
                    )
                )
                initiator_counts[host] += 1
                next_conversation_attempt[host] = time + float(rng.exponential(cfg.mean_conversation_interval))

            if next_conversation_attempt[host] <= time + 1e-9:
                next_conversation_attempt[host] = time + float(rng.exponential(cfg.mean_conversation_interval))

        survivors: list[Conversation] = []

        for conv in active:
            generated = min(conv.remaining_packets, int(rng.poisson(conv.rate * cfg.step)))

            if generated <= 0:
                survivors.append(conv)
                continue

            source = conv.initiator
            target = conv.partner
            optimal_route = shortest_path(graph, source, target)
            route = conv.route

            if route is None:
                cached = caches[source].path(source, target, time)
                route = tuple(cached) if cached is not None else None

            if route is None and time >= next_discovery_time[source, target]:
                route, control = perform_route_discovery(source, target, graph, caches, time, cfg)
                total_tx += control

                if route is None:
                    next_discovery_time[source, target] = time + request_period[source, target]
                    request_period[source, target] = min(request_period[source, target] * 2.0, cfg.max_request_period)
                else:
                    next_discovery_time[source, target] = 0.0
                    request_period[source, target] = cfg.nonpropagating_request_period

            if route is not None and route_valid(route, graph):
                hop_count = len(route) - 1
                total_tx += 2.0 * generated * hop_count * EXPECTED_UNICAST_ATTEMPTS
                used_hops += 2.0 * generated * hop_count

                if optimal_route is not None:
                    optimal_hop_count = len(optimal_route) - 1
                    optimal_tx += 2.0 * generated * optimal_hop_count
                    optimal_hops += 2.0 * generated * optimal_hop_count

                conv.remaining_packets -= generated
                conv.route = route

                for node in route:
                    caches[node].add_path(route, time, cfg)

            elif route is not None:
                broken_hop_index = next(
                    i for i in range(len(route) - 1) if route[i + 1] not in graph[route[i]]
                )
                route_hop_count = len(route) - 1

                # One packet triggers the failure, and the detecting node returns a route error upstream.
                total_tx += broken_hop_index * EXPECTED_UNICAST_ATTEMPTS + 3.0
                if broken_hop_index > 0:
                    total_tx += broken_hop_index * EXPECTED_UNICAST_ATTEMPTS

                used_hops += route_hop_count

                if optimal_route is not None:
                    optimal_hop_count = len(optimal_route) - 1
                    optimal_tx += optimal_hop_count
                    optimal_hops += optimal_hop_count

                failed_edge = (route[broken_hop_index], route[broken_hop_index + 1])
                for cache in caches:
                    cache.remove_edge(*failed_edge)

                conv.remaining_packets -= 1
                conv.route = None
                remaining_after_failure = generated - 1

                if remaining_after_failure > 0 and time >= next_discovery_time[source, target]:
                    new_route, control = perform_route_discovery(source, target, graph, caches, time, cfg)
                    total_tx += control

                    if new_route is not None and route_valid(new_route, graph):
                        new_hop_count = len(new_route) - 1
                        total_tx += 2.0 * remaining_after_failure * new_hop_count * EXPECTED_UNICAST_ATTEMPTS
                        used_hops += 2.0 * remaining_after_failure * new_hop_count

                        if optimal_route is not None:
                            optimal_hop_count = len(optimal_route) - 1
                            optimal_tx += 2.0 * remaining_after_failure * optimal_hop_count
                            optimal_hops += 2.0 * remaining_after_failure * optimal_hop_count

                        conv.remaining_packets -= remaining_after_failure
                        conv.route = new_route
                        next_discovery_time[source, target] = 0.0
                        request_period[source, target] = cfg.nonpropagating_request_period
                    else:
                        next_discovery_time[source, target] = time + request_period[source, target]
                        request_period[source, target] = min(
                            request_period[source, target] * 2.0, cfg.max_request_period
                        )

            if conv.remaining_packets > 0:
                survivors.append(conv)

        active = survivors

        for node in nodes:
            node.advance(cfg.step, pause_time, rng, cfg)

        time += cfg.step

    if optimal_tx == 0.0 or optimal_hops == 0.0:
        return float("nan"), float("nan")

    return total_tx / optimal_tx, used_hops / optimal_hops


def run_sweep(cfg: EvalConfig) -> dict[str, np.ndarray]:
    """Evaluate the aggregate model over every host-count and pause-time combination."""

    overhead = np.zeros((len(HOST_COUNTS), len(PAUSE_TIMES)), dtype=float)
    route_ratio = np.zeros_like(overhead)

    print("Running Figure 5 / Figure 7 aggregate sweep (this may take several minutes)...")

    for i, host_count in enumerate(HOST_COUNTS):
        for j, pause_time in enumerate(PAUSE_TIMES):
            sample_array = np.array(
                [run_trial(int(host_count), float(pause_time), seed, cfg) for seed in range(cfg.trials)],
                dtype=float,
            )
            valid = int(np.sum(~np.isnan(sample_array[:, 0])))
            overhead[i, j] = float(np.nanmean(sample_array[:, 0]))
            route_ratio[i, j] = float(np.nanmean(sample_array[:, 1]))
            print(
                f"hosts={int(host_count):2d}, pause={int(pause_time):4d}, "
                f"overhead={overhead[i, j]:.3f}, route_ratio={route_ratio[i, j]:.3f}, "
                f"valid_trials={valid}/{cfg.trials}"
            )

    return {"overhead": overhead, "route_ratio": route_ratio}


In [ ]:
evaluation = run_sweep(SIM_CONFIG)
evaluation["overhead"], evaluation["route_ratio"]


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))

for row, host_count in enumerate(HOST_COUNTS):
    ax.plot(
        PAUSE_TIMES,
        PAPER_FIGURE5[row],
        color="black",
        linestyle=LINESTYLES[int(host_count)],
        marker=MARKERS[int(host_count)],
        markersize=5,
        markerfacecolor="white",
        alpha=0.8,
    )
    ax.plot(
        PAUSE_TIMES,
        evaluation["overhead"][row],
        color=COLORS[int(host_count)],
        linestyle="-",
        linewidth=2,
        marker=MARKERS[int(host_count)],
        markersize=5.5,
        label=f"{int(host_count)} hosts",
    )

ax.text(
    0.02,
    0.98,
    "Black dashed/open markers: approximate paper digitization\nColored solid markers: aggregate reproduction",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#d9d9d9"),
)
ax.set_xlabel("Pause Time (s)")
ax.set_ylabel("Num transmits / Optimal num transmits")
ax.set_title("Figure 5 Reproduction: Average Total Number of Transmissions Relative to Optimal")
ax.set_xlim(-40, 4080)
ax.set_ylim(0.95, 2.75)
ax.legend(title="Network size", loc="upper right")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))

figure5_error = evaluation["overhead"] - PAPER_FIGURE5
im = ax.imshow(np.abs(figure5_error), cmap="jet", aspect="auto", interpolation="nearest")
ax.set_xticks(np.arange(len(PAUSE_TIMES)), labels=[str(int(value)) for value in PAUSE_TIMES], rotation=45)
ax.set_yticks(np.arange(len(HOST_COUNTS)), labels=[str(int(value)) for value in HOST_COUNTS])
ax.set_xlabel("Pause Time (s)")
ax.set_ylabel("Host Count")
ax.set_title("Figure 5 Absolute Error Heatmap")

for row in range(figure5_error.shape[0]):
    for col in range(figure5_error.shape[1]):
        ax.text(col, row, f"{figure5_error[row, col]:+.2f}", ha="center", va="center", fontsize=8, color="white")

plt.colorbar(im, ax=ax, label="Absolute ratio error")
plt.show()


## Figure 6 Explanation

Figure 6 is not an aggregate metric.
It is a **single representative placement** of 24 hosts whose instantaneous communication graph splits into two disconnected components.
Because the paper does not give the seed or coordinates, the reproduction searches deterministic seeds and returns the first static placement with exactly two connected components.


In [ ]:
def connected_components(graph: list[list[int]]) -> list[list[int]]:
    """Enumerate connected components of an undirected graph."""

    seen: set[int] = set()
    components: list[list[int]] = []

    for node in range(len(graph)):
        if node in seen:
            continue

        queue = deque([node])
        seen.add(node)
        component: list[int] = []

        while queue:
            current = queue.popleft()
            component.append(current)

            for neighbor in graph[current]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    queue.append(neighbor)

        components.append(component)

    return components


def find_two_component_layout(host_count: int = 24, max_seed: int = 5000) -> tuple[np.ndarray, list[list[int]], int]:
    """Search deterministic seeds for the first placement with exactly two components."""

    for seed in range(max_seed):
        rng = np.random.default_rng(seed)
        points = rng.uniform(0.0, SIM_CONFIG.area_size, size=(host_count, 2))
        graph = [[] for _ in range(host_count)]

        for i in range(host_count):
            for j in range(i + 1, host_count):
                if float(np.linalg.norm(points[i] - points[j])) <= SIM_CONFIG.radio_range:
                    graph[i].append(j)
                    graph[j].append(i)

        components = connected_components(graph)
        if len(components) == 2:
            return points, graph, seed

    raise RuntimeError("No two-component placement was found in the searched seed range.")


figure6_points, figure6_graph, figure6_seed = find_two_component_layout()
figure6_components = connected_components(figure6_graph)
print("Representative seed:", figure6_seed)
print("Component sizes:", [len(component) for component in figure6_components])

fig, ax = plt.subplots(figsize=(6.2, 6.2))
for node, neighbors in enumerate(figure6_graph):
    x0, y0 = figure6_points[node]
    for neighbor in neighbors:
        if neighbor > node:
            x1, y1 = figure6_points[neighbor]
            ax.plot([x0, x1], [y0, y1], color="black", linewidth=0.8, alpha=0.9)

ax.scatter(figure6_points[:, 0], figure6_points[:, 1], s=36, color="black")
ax.set_xlim(0.0, SIM_CONFIG.area_size)
ax.set_ylim(0.0, SIM_CONFIG.area_size)
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Figure 6 Reproduction: Example of Disconnected Clusters with 24 Hosts")
plt.show()


## Figure 7 Explanation

Figure 7 measures how close DSR stays to hop-optimal routing while nodes move.
The paper reports that the difference from optimal is negligible:
most curves stay within about $1.01$ of optimal and all within about $1.09$.

Expected qualitative behavior:

- every curve should stay close to $1$,
- mobility should increase the ratio slightly,
- denser networks should stay especially close to optimal because the cache offers many short alternatives.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2))

for row, host_count in enumerate(HOST_COUNTS):
    ax.plot(
        PAUSE_TIMES,
        PAPER_FIGURE7[row],
        color="black",
        linestyle=LINESTYLES[int(host_count)],
        marker=MARKERS[int(host_count)],
        markersize=5,
        markerfacecolor="white",
        alpha=0.8,
    )
    ax.plot(
        PAUSE_TIMES,
        evaluation["route_ratio"][row],
        color=COLORS[int(host_count)],
        linestyle="-",
        linewidth=2,
        marker=MARKERS[int(host_count)],
        markersize=5.5,
        label=f"{int(host_count)} hosts",
    )

ax.text(
    0.02,
    0.98,
    "Black dashed/open markers: approximate paper digitization\nColored solid markers: aggregate reproduction",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#d9d9d9"),
)
ax.set_xlabel("Pause Time (s)")
ax.set_ylabel("Average route length / Optimal route length")
ax.set_title("Figure 7 Reproduction: Average Route Length Used Relative to Optimal")
ax.set_xlim(-40, 4080)
ax.set_ylim(0.995, 1.125)
ax.legend(title="Network size", loc="upper right")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))

figure7_error = evaluation["route_ratio"] - PAPER_FIGURE7
im = ax.imshow(np.abs(figure7_error), cmap="jet", aspect="auto", interpolation="nearest")
ax.set_xticks(np.arange(len(PAUSE_TIMES)), labels=[str(int(value)) for value in PAUSE_TIMES], rotation=45)
ax.set_yticks(np.arange(len(HOST_COUNTS)), labels=[str(int(value)) for value in HOST_COUNTS])
ax.set_xlabel("Pause Time (s)")
ax.set_ylabel("Host Count")
ax.set_title("Figure 7 Absolute Error Heatmap")

for row in range(figure7_error.shape[0]):
    for col in range(figure7_error.shape[1]):
        ax.text(col, row, f"{figure7_error[row, col]:+.3f}", ha="center", va="center", fontsize=8, color="white")

plt.colorbar(im, ax=ax, label="Absolute ratio error")
plt.show()


## Summary

| Figure / Section | What it shows | Status |
|---|---|---|
| Figure 1 | Minimal three-host relay topology | Reproduced schematically |
| Figure 2 | Route-cache reuse and cache-based reply idea | Reproduced schematically |
| Figure 3 | Route shortening by promiscuous overhearing | Reproduced schematically |
| Figure 4 | Route error returned to the source | Reproduced schematically |
| Figure 5 | Normalized transmission overhead vs pause time and host count | Reproduced approximately with an aggregate simulator and compared against digitized paper curves |
| Figure 6 | Representative disconnected placement with 24 hosts | Reproduced qualitatively by deterministic seed search |
| Figure 7 | Average route length relative to optimal | Reproduced approximately with the same aggregate simulator and compared against digitized paper curves |

The qualitative conclusions of the paper are preserved:
DSR becomes nearly overhead-free as mobility slows,
disconnected sparse networks remain somewhat expensive,
and the routes used stay very close to hop-optimal even without the route-shortening optimization.
